# orchestra-smoke — kiểm CLI điều phối nhiều acc bằng kernel CPU

Không chạm GPU, không chạm dữ liệu thi. Mỗi shard ghi một dòng cho mỗi câu trong
`[tu, den)`, rồi ghi mốc bằng CHÍNH `kaggle_orchestra.ghi_moc` trong src đã đẩy — tức
kiểm luôn đường src → notebook → mốc → tải về → ghép. `loi_o_shard` cố ý ném lỗi để kiểm
nhánh HỎNG với log Kaggle thật.

In [ ]:
# ORCHESTRA-PARAMS — kaggle_orchestra.py thay cell này lúc đẩy. Giá trị dưới chỉ để chạy tay.
SHARD = {"job": "tay", "chi_so": 0, "so_shard": 1, "tu": 0, "den": 4, "commit": "", "noi_tiep": None, "output": "x.jsonl"}

In [ ]:
import os, sys, shutil, subprocess, json
from pathlib import Path

# Codebase chép ra /tmp chứ KHÔNG ra /kaggle/working: mọi thứ trong working là output, và
# orchestrator tải output của từng shard về máy dev sau mỗi lần dừng — 30 MB repo × số lần.
SRC = next((c.parent.parent.parent for c in Path("/kaggle/input").glob("**/legalqa/bm25.py")
            if "legalqa-v10-src" in str(c)), None)
assert SRC, "Không thấy codebase — gắn dataset legalqa-v10-src"
REPO = Path("/tmp/repo")
shutil.rmtree(REPO, ignore_errors=True)
shutil.copytree(SRC, REPO)
os.chdir(REPO)
sys.path.insert(0, "src"); sys.path.insert(0, "scripts")
commit = (REPO / "COMMIT.txt").read_text().strip()
print("codebase:", commit, "| shard", SHARD)
# Mọi shard phải chạy CÙNG commit: dataset src của từng acc được đẩy riêng, lệch một acc
# là ghép hai phiên bản code vào một tập mà không dấu vết.
assert not SHARD["commit"] or commit == SHARD["commit"], f"src {commit} != {SHARD['commit']}"

OUT = Path("/kaggle/working/out"); OUT.mkdir(parents=True, exist_ok=True)
# Nối tiếp: dataset `orch-<job>-s<i>` mang phần đã làm từ acc trước.
if SHARD["noi_tiep"]:
    for f in Path("/kaggle/input").glob("**/*.jsonl"):
        if SHARD["noi_tiep"] in str(f):
            shutil.copy(f, OUT / f.name)
            print(f"nối tiếp {f.name}: {sum(1 for _ in open(OUT / f.name, encoding='utf-8')):,} dòng")

In [ ]:
from kaggle_orchestra import doc_ids, ghi_moc

ra = OUT / SHARD["output"]
xong = set(doc_ids(ra))
with ra.open("a", encoding="utf-8") as f:
    for k in range(SHARD["tu"], SHARD["den"]):
        if f"q{k}" in xong:
            continue
        if SHARD.get("loi_o_shard") == SHARD["chi_so"] and k == SHARD["tu"] + 1:
            f.flush()
            raise KeyError(f"lỗi cố ý ở câu q{k} để kiểm nhánh HỎNG")
        f.write(json.dumps({"id": f"q{k}", "du_doan": f"đáp án {k}", "commit": commit}) + "\n")
print(ghi_moc(ra, SHARD["tu"], SHARD["den"]))